# Aula 18 - Notebook: Grafos de Logística de Grãos, Processo e Produtos Acabados

**Projeto:** SCADA-Core Automática — GRUPO 4
**Unidade Fabril:** Linha Automatizada de Seleção, Pesagem Dinâmica e Classificação Óptica de Grãos com Separação Pneumática Tripla
**Engenharia de Automação e Controle** — ECAA08

---

### 1. Objetivos do Notebook
Este notebook complementa a malha de processo das Aulas 11 a 15 com as redes logísticas do layout `Grafo_Logistica.png`. Modelamos a mesma planta física como uma **rede multicamada**:

| Camada | Grafo | Peso | Pergunta respondida |
| --- | --- | --- | --- |
| Fluxo de materiais | $G_M$ (dirigido) | distância interna estimada (m) | Como o arroz que chega no caminhão alcança a doca como produto acabado? |
| Circulação de veículos | $G_V$ (dirigido, anel viário) | distância de circulação estimada (m) | Por onde o caminhão graneleiro ou de expedição pode trafegar? |
| Alocação de estoque | $G_E$ (dirigido) | deslocamento de empilhadeira (m) | Em qual posição do Galpão B armazenar cada palete? |

Algoritmos aplicados: **Dijkstra** (Aula 14), **BFS de alcançabilidade** (Aula 13), **ordenação topológica de Kahn** com detecção de ciclo por **DFS**, análise de **arcos críticos** (falha única, Aula 15) e roteamento com **paradas obrigatórias**.

> **Nota metodológica:** os pesos fora da linha de classificação são estimativas didáticas para comparação de rotas; não são medidas da imagem. Os pesos da linha `FN-101` $\to$ silos são exatamente os comprimentos da Aula 11.

### 2. Funções de Formatação Tabular em ASCII Puro

In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

### 3. Classe `GrafoLogistico`
Representação por **lista de adjacência** (grafo esparso). Cada arco guarda destino, peso e descrição operacional.

In [2]:
from collections import deque
from dataclasses import dataclass
from heapq import heappop, heappush
from math import inf
from typing import Dict, List, Optional, Tuple


@dataclass(frozen=True)
class Rota:
    destino: str
    peso: float
    descricao: str


class GrafoLogistico:
    def __init__(self, nome: str) -> None:
        self.nome = nome
        self._adjacencias: Dict[str, List[Rota]] = {}

    # ------------------------------------------------------------ estrutura
    def adicionar_rota(self, origem: str, destino: str, peso: float, descricao: str) -> None:
        if peso < 0:
            raise ValueError("Dijkstra exige pesos não negativos.")
        self._adjacencias.setdefault(origem, []).append(Rota(destino, peso, descricao))
        self._adjacencias.setdefault(destino, [])

    def remover_rota(self, origem: str, destino: str) -> Rota:
        for rota in self._adjacencias[origem]:
            if rota.destino == destino:
                self._adjacencias[origem].remove(rota)
                return rota
        raise KeyError(f"Arco {origem} -> {destino} não existe em {self.nome}.")

    def copia(self) -> "GrafoLogistico":
        g = GrafoLogistico(self.nome)
        g._adjacencias = {v: list(rotas) for v, rotas in self._adjacencias.items()}
        return g

    @property
    def vertices(self) -> List[str]:
        return list(self._adjacencias)

    def arcos(self) -> List[Tuple[str, Rota]]:
        return [(u, r) for u, rotas in self._adjacencias.items() for r in rotas]

    def peso(self, origem: str, destino: str) -> float:
        return min(r.peso for r in self._adjacencias[origem] if r.destino == destino)

    # ------------------------------------------------------------ Dijkstra
    def menor_caminho(self, origem: str, destino: str) -> Tuple[float, List[str]]:
        if origem not in self._adjacencias or destino not in self._adjacencias:
            raise KeyError(f"Origem e destino devem pertencer ao grafo {self.nome}.")

        distancias = {vertice: inf for vertice in self._adjacencias}
        predecessores: Dict[str, str] = {}
        distancias[origem] = 0.0
        fila: List[Tuple[float, str]] = [(0.0, origem)]

        while fila:
            distancia_atual, atual = heappop(fila)
            if distancia_atual != distancias[atual]:
                continue
            if atual == destino:
                break
            for rota in self._adjacencias[atual]:
                candidata = distancia_atual + rota.peso
                if candidata < distancias[rota.destino]:
                    distancias[rota.destino] = candidata
                    predecessores[rota.destino] = atual
                    heappush(fila, (candidata, rota.destino))

        if distancias[destino] == inf:
            raise ValueError(f"Não existe rota de {origem} para {destino}.")

        caminho = [destino]
        while caminho[-1] != origem:
            caminho.append(predecessores[caminho[-1]])
        caminho.reverse()
        return distancias[destino], caminho

    def rota_com_paradas(self, paradas: List[str]) -> Tuple[float, List[str]]:
        """Encadeia Dijkstra entre paradas obrigatórias (ex.: calador, moega, saída)."""
        total, caminho = 0.0, [paradas[0]]
        for a, b in zip(paradas, paradas[1:]):
            d, trecho = self.menor_caminho(a, b)
            total += d
            caminho += trecho[1:]
        return total, caminho

    # ------------------------------------------------- BFS e ordenação topológica
    def alcancaveis(self, origem: str) -> set:
        vistos, fila = {origem}, deque([origem])
        while fila:
            u = fila.popleft()
            for rota in self._adjacencias[u]:
                if rota.destino not in vistos:
                    vistos.add(rota.destino)
                    fila.append(rota.destino)
        return vistos

    def ordenacao_topologica(self) -> Tuple[List[str], List[str]]:
        """Algoritmo de Kahn em O(|V| + |E|). Retorna (ordem, vertices_nao_ordenados)."""
        grau_entrada = {v: 0 for v in self._adjacencias}
        for _, rota in self.arcos():
            grau_entrada[rota.destino] += 1
        fila = deque(v for v in self._adjacencias if grau_entrada[v] == 0)
        ordem = []
        while fila:
            u = fila.popleft()
            ordem.append(u)
            for rota in self._adjacencias[u]:
                grau_entrada[rota.destino] -= 1
                if grau_entrada[rota.destino] == 0:
                    fila.append(rota.destino)
        restantes = [v for v in self._adjacencias if v not in ordem]
        return ordem, restantes

    def encontrar_ciclo(self) -> Optional[List[str]]:
        """DFS com cores: a primeira aresta de retorno (back edge) fecha um ciclo."""
        cor = {v: "branco" for v in self._adjacencias}
        pilha: List[str] = []

        def visitar(u: str) -> Optional[List[str]]:
            cor[u] = "cinza"
            pilha.append(u)
            for rota in self._adjacencias[u]:
                if cor[rota.destino] == "cinza":
                    return pilha[pilha.index(rota.destino):] + [rota.destino]
                if cor[rota.destino] == "branco":
                    ciclo = visitar(rota.destino)
                    if ciclo:
                        return ciclo
            pilha.pop()
            cor[u] = "preto"
            return None

        for v in self._adjacencias:
            if cor[v] == "branco":
                ciclo = visitar(v)
                if ciclo:
                    return ciclo
        return None

    def sequencia_fisicamente_valida(self, sequencia: List[str]) -> bool:
        """Uma sequência é inválida se algum passo posterior antecede fisicamente um anterior."""
        for i, anterior in enumerate(sequencia):
            for posterior in sequencia[i + 1:]:
                if anterior in self.alcancaveis(posterior) and anterior != posterior:
                    return False
        return True

    # ------------------------------------------------------- robustez (Aula 15)
    def arcos_criticos(self, origem: str, destino: str) -> List[Tuple[str, Rota]]:
        """Arcos cuja falha isolada desconecta destino de origem."""
        criticos = []
        for u, rota in self.arcos():
            teste = self.copia()
            teste.remover_rota(u, rota.destino)
            if destino not in teste.alcancaveis(origem):
                criticos.append((u, rota))
        return criticos


def adicionar_rotas(grafo: GrafoLogistico, rotas: List[Tuple[str, str, float, str]]) -> None:
    for origem, destino, peso, descricao in rotas:
        grafo.adicionar_rota(origem, destino, peso, descricao)

print("Classe GrafoLogistico carregada.")

Classe GrafoLogistico carregada.


### 4. $G_M$ — Grafo Dirigido do Fluxo de Materiais
O grafo é montado em três blocos: **recebimento** (do caminhão graneleiro até o funil `FN-101`), **linha de classificação** (os 13 arcos produtivos da Aula 11, com os mesmos comprimentos) e **expedição** (dos silos até as docas). O arco de reciclo $e_{14}$ (`SILO-702` $\to$ `FN-101`, comporta `XV-104`) é opcional e será analisado na Seção 6.

In [3]:
ROTAS_RECEBIMENTO = [
    ("Portaria", "Balança de entrada", 30, "liberação documental e pesagem bruta do graneleiro"),
    ("Balança de entrada", "Calador / amostragem", 25, "amostra do lote pelo calador pneumático"),
    ("Calador / amostragem", "Moega MG-001", 60, "lote liberado pelo LAB-CQ (umidade e impurezas)"),
    ("Moega MG-001", "Silo pulmão SP-001", 35, "transportador de corrente TC-001"),
    ("Moega MG-001", "Silo pulmão SP-002", 42, "transportador de corrente TC-002"),
    ("Silo pulmão SP-001", "FN-101", 30, "elevador de canecas EL-001"),
    ("Silo pulmão SP-002", "FN-101", 20, "elevador de canecas EL-002"),
]

# Linha de classificação: arcos e1..e13 e comprimentos da Aula 11
ROTAS_LINHA = [
    ("FN-101", "ALIM-101", 1.2, "e1: duto de carga (XV-101)"),
    ("ALIM-101", "EST-201", 1.8, "e2: calha vibratória"),
    ("EST-201", "BAL-301", 2.5, "e3: correia até a balança dinâmica"),
    ("BAL-301", "VIS-401", 3.0, "e4: correia até o túnel óptico"),
    ("VIS-401", "DIV-501", 1.0, "e5: trigger óptico"),
    ("DIV-501", "EJE-602", 2.0, "e6: correia sincronizada"),
    ("EJE-602", "CALHA-602", 0.8, "e7: sopro pneumático B (FY-602)"),
    ("EJE-602", "EJE-603", 1.5, "e8: continuidade da esteira"),
    ("EJE-603", "CALHA-603", 0.8, "e9: sopro pneumático C (FY-603)"),
    ("EJE-603", "CALHA-701", 2.2, "e10: descarga terminal"),
    ("CALHA-701", "SILO-701", 3.5, "e11: duto gravitacional (XV-701)"),
    ("CALHA-602", "SILO-702", 4.0, "e12: calha inclinada (XV-702)"),
    ("CALHA-603", "SILO-703", 3.2, "e13: calha de rejeito (XV-703)"),
]
ROTA_RECICLO = ("SILO-702", "FN-101", 12.0, "e14: elevador de reciclo / retrabalho (XV-104)")

ROTAS_EXPEDICAO = [
    ("SILO-701", "Empacotadora EMP-801", 18, "pacotes de 1 kg e 5 kg — arroz Tipo 1"),
    ("Empacotadora EMP-801", "Paletizadora PAL-801", 12, "enfardamento e paletização"),
    ("Paletizadora PAL-801", "Estoque Tipo 1", 30, "endereçamento no Galpão B"),
    ("Estoque Tipo 1", "Docas de expedição", 25, "separação de pedidos"),
    ("SILO-702", "Ensacadora BIG-802", 20, "big bags de 1 t — arroz comercial"),
    ("Ensacadora BIG-802", "Estoque Tipo 2", 26, "endereçamento no Galpão B"),
    ("Estoque Tipo 2", "Docas de expedição", 22, "separação de pedidos"),
    ("SILO-703", "Caçamba CAC-803", 15, "quirera e impurezas — rosca de descarga"),
    ("Caçamba CAC-803", "Saída de subprodutos", 40, "ração animal / compostagem (rota segregada)"),
]

def criar_fluxo_materiais(incluir_reciclo: bool = False) -> GrafoLogistico:
    g = GrafoLogistico("G_M (fluxo de materiais)")
    adicionar_rotas(g, ROTAS_RECEBIMENTO + ROTAS_LINHA + ROTAS_EXPEDICAO)
    if incluir_reciclo:
        g.adicionar_rota(*ROTA_RECICLO)
    return g

fluxo_materiais = criar_fluxo_materiais()
print(f"{fluxo_materiais.nome}: |V| = {len(fluxo_materiais.vertices)} vértices, |E| = {len(fluxo_materiais.arcos())} arcos")

distancia, caminho = fluxo_materiais.menor_caminho("Portaria", "Docas de expedição")
print("\nFluxo de materiais até a expedição (rota de referência mais curta)")
print("  " + " -> ".join(caminho))
print(f"  Distância didática total: {distancia:.1f} m")

G_M (fluxo de materiais): |V| = 28 vértices, |E| = 29 arcos

Fluxo de materiais até a expedição (rota de referência mais curta)
  Portaria -> Balança de entrada -> Calador / amostragem -> Moega MG-001 -> Silo pulmão SP-002 -> FN-101 -> ALIM-101 -> EST-201 -> BAL-301 -> VIS-401 -> DIV-501 -> EJE-602 -> CALHA-602 -> SILO-702 -> Ensacadora BIG-802 -> Estoque Tipo 2 -> Docas de expedição
  Distância didática total: 261.3 m


#### 4.1. Rotas de referência por produto acabado
Uma rota em $G_M$ é uma **rota operacional de referência**: a categoria de um grão é decidida pela visão computacional (`VIS-401`), não pelo Dijkstra. Por isso calculamos a rota até **cada destino**, e não apenas a rota mais curta até as docas.

In [4]:
destinos = ["Estoque Tipo 1", "Estoque Tipo 2", "Docas de expedição", "Saída de subprodutos"]
tabela_destinos = []
for destino in destinos:
    d, c = fluxo_materiais.menor_caminho("Portaria", destino)
    silo = next((v for v in c if v.startswith("SILO")), "-")
    tabela_destinos.append({
        "Destino": destino,
        "Silo de origem": silo,
        "Pulmão usado": next(v for v in c if v.startswith("Silo pulmão")),
        "Arcos": len(c) - 1,
        "Distância (m)": f"{d:.1f}",
    })
print(formatar_tabela(tabela_destinos))

print("\nComparação dos dois silos pulmão (soma acumulada, não apenas o primeiro arco):")
for pulmao in ["Silo pulmão SP-001", "Silo pulmão SP-002"]:
    w1 = fluxo_materiais.peso("Moega MG-001", pulmao)
    w2 = fluxo_materiais.peso(pulmao, "FN-101")
    print(f"  Moega -> {pulmao:<19} -> FN-101: {w1:.0f} + {w2:.0f} = {w1 + w2:.0f} m")

Destino              | Silo de origem | Pulmão usado       | Arcos | Distância (m)
---------------------+----------------+--------------------+-------+--------------
Estoque Tipo 1       | SILO-701       | Silo pulmão SP-002 | 17    | 255.7        
Estoque Tipo 2       | SILO-702       | Silo pulmão SP-002 | 15    | 239.3        
Docas de expedição   | SILO-702       | Silo pulmão SP-002 | 16    | 261.3        
Saída de subprodutos | SILO-703       | Silo pulmão SP-002 | 16    | 249.0        

Comparação dos dois silos pulmão (soma acumulada, não apenas o primeiro arco):
  Moega -> Silo pulmão SP-001  -> FN-101: 35 + 30 = 65 m
  Moega -> Silo pulmão SP-002  -> FN-101: 42 + 20 = 62 m


### 5. Segregação Sanitária: Rejeito Nunca Alcança Produto Acabado
Requisito de segurança de alimentos: o material da Categoria C (`SILO-703`) não pode ter **nenhum caminho dirigido** até estoques de produto ou docas. Verificamos por BFS (Aula 13).

In [5]:
alcance_rejeito = fluxo_materiais.alcancaveis("SILO-703")
proibidos = ["Estoque Tipo 1", "Estoque Tipo 2", "Docas de expedição", "SILO-701", "SILO-702"]
tabela_seg = [{"Vértice protegido": v, "Alcançável a partir de SILO-703?": "SIM (FALHA)" if v in alcance_rejeito else "não"}
              for v in proibidos]
print(formatar_tabela(tabela_seg))
print(f"\nAlcance de SILO-703: {sorted(alcance_rejeito)}")

Vértice protegido  | Alcançável a partir de SILO-703?
-------------------+---------------------------------
Estoque Tipo 1     | não                             
Estoque Tipo 2     | não                             
Docas de expedição | não                             
SILO-701           | não                             
SILO-702           | não                             

Alcance de SILO-703: ['Caçamba CAC-803', 'SILO-703', 'Saída de subprodutos']


### 6. $G_M$ é um DAG? Ordenação Topológica (Kahn) e o Arco de Reciclo $e_{14}$
* **Sem reciclo:** cada etapa é irreversível; o grafo deve ser um DAG e admitir ordenação topológica.
* **Com reciclo ativo** (`XV-104` aberta): o arco $e_{14}$ cria o ciclo identificado nas Aulas 11 a 13. O algoritmo de Kahn deixa vértices sem ordenar, e a DFS devolve o ciclo explícito.

In [6]:
ordem, restantes = fluxo_materiais.ordenacao_topologica()
print(">>> G_M sem reciclo (XV-104 fechada)")
print(f"É DAG? {len(restantes) == 0} | vértices ordenados: {len(ordem)} de {len(fluxo_materiais.vertices)}")
print("Ordenação topológica (sequência de partida/auditoria):")
for i, v in enumerate(ordem, 1):
    print(f"  {i:>2}. {v}")

fluxo_reciclo = criar_fluxo_materiais(incluir_reciclo=True)
ordem_r, restantes_r = fluxo_reciclo.ordenacao_topologica()
ciclo = fluxo_reciclo.encontrar_ciclo()
print("\n>>> G_M com reciclo ativo (XV-104 aberta)")
print(f"É DAG? {len(restantes_r) == 0} | Kahn ordenou {len(ordem_r)} vértices e travou em {len(restantes_r)}")
print(f"Ciclo encontrado pela DFS: {' -> '.join(ciclo)}")
print("Interpretação: o retrabalho exige uma regra explícita de liberação; o fluxo normal deve ser auditado sem e14.")

>>> G_M sem reciclo (XV-104 fechada)
É DAG? True | vértices ordenados: 28 de 28
Ordenação topológica (sequência de partida/auditoria):
   1. Portaria
   2. Balança de entrada
   3. Calador / amostragem
   4. Moega MG-001
   5. Silo pulmão SP-001
   6. Silo pulmão SP-002
   7. FN-101
   8. ALIM-101
   9. EST-201
  10. BAL-301
  11. VIS-401
  12. DIV-501
  13. EJE-602
  14. CALHA-602
  15. EJE-603
  16. SILO-702
  17. CALHA-603
  18. CALHA-701
  19. Ensacadora BIG-802
  20. SILO-703
  21. SILO-701
  22. Estoque Tipo 2
  23. Caçamba CAC-803
  24. Empacotadora EMP-801
  25. Saída de subprodutos
  26. Paletizadora PAL-801
  27. Estoque Tipo 1
  28. Docas de expedição

>>> G_M com reciclo ativo (XV-104 aberta)
É DAG? False | Kahn ordenou 6 vértices e travou em 22
Ciclo encontrado pela DFS: FN-101 -> ALIM-101 -> EST-201 -> BAL-301 -> VIS-401 -> DIV-501 -> EJE-602 -> CALHA-602 -> SILO-702 -> FN-101
Interpretação: o retrabalho exige uma regra explícita de liberação; o fluxo normal deve ser audi

#### 6.1. Auditoria de sequências de produção
Uma sequência registrada pelo SCADA é fisicamente possível somente se nenhum passo posterior for antecessor de um passo anterior no DAG.

In [7]:
sequencias = {
    "Recebimento -> classificação -> empacotamento": ["Moega MG-001", "FN-101", "VIS-401", "SILO-701", "Empacotadora EMP-801"],
    "Empacotar antes da inspeção óptica":            ["FN-101", "Empacotadora EMP-801", "VIS-401"],
    "Big bag a partir do silo comercial":            ["SILO-702", "Ensacadora BIG-802", "Estoque Tipo 2"],
    "Paletizar e depois pesar na BAL-301":           ["Paletizadora PAL-801", "BAL-301"],
}
print(formatar_tabela([{"Sequência auditada": nome, "Passos": " -> ".join(seq),
                        "Fisicamente válida?": fluxo_materiais.sequencia_fisicamente_valida(seq)}
                       for nome, seq in sequencias.items()]))

Sequência auditada                            | Passos                                                                | Fisicamente válida?
----------------------------------------------+-----------------------------------------------------------------------+--------------------
Recebimento -> classificação -> empacotamento | Moega MG-001 -> FN-101 -> VIS-401 -> SILO-701 -> Empacotadora EMP-801 | True               
Empacotar antes da inspeção óptica            | FN-101 -> Empacotadora EMP-801 -> VIS-401                             | False              
Big bag a partir do silo comercial            | SILO-702 -> Ensacadora BIG-802 -> Estoque Tipo 2                      | True               
Paletizar e depois pesar na BAL-301           | Paletizadora PAL-801 -> BAL-301                                       | False              


### 7. Arcos Críticos: Falha Única que Interrompe a Expedição
Para cada arco de $G_M$, removemos o arco e testamos se `Docas de expedição` continua alcançável a partir da `Portaria`. Os arcos críticos são candidatos a redundância (como os dois silos pulmão) ou a estoque de peças sobressalentes.

In [8]:
criticos = fluxo_materiais.arcos_criticos("Portaria", "Docas de expedição")
print(formatar_tabela([{"Arco crítico": f"{u} -> {r.destino}", "Descrição": r.descricao} for u, r in criticos]))
print(f"\n{len(criticos)} de {len(fluxo_materiais.arcos())} arcos são pontos únicos de falha para a expedição.")

teste = fluxo_materiais.copia()
teste.remover_rota("Moega MG-001", "Silo pulmão SP-002")
d_alt, c_alt = teste.menor_caminho("Portaria", "Docas de expedição")
print(f"\nFalha no transportador TC-002 (Moega -> SP-002): desvio automático via SP-001")
print(f"  Nova rota: {' -> '.join(c_alt[:6])} -> ...")
print(f"  Distância: {d_alt:.1f} m (+{d_alt - distancia:.1f} m em relação à rota nominal)")

Arco crítico                               | Descrição                                         
-------------------------------------------+---------------------------------------------------
Portaria -> Balança de entrada             | liberação documental e pesagem bruta do graneleiro
Balança de entrada -> Calador / amostragem | amostra do lote pelo calador pneumático           
Calador / amostragem -> Moega MG-001       | lote liberado pelo LAB-CQ (umidade e impurezas)   
FN-101 -> ALIM-101                         | e1: duto de carga (XV-101)                        
ALIM-101 -> EST-201                        | e2: calha vibratória                              
EST-201 -> BAL-301                         | e3: correia até a balança dinâmica                
BAL-301 -> VIS-401                         | e4: correia até o túnel óptico                    
VIS-401 -> DIV-501                         | e5: trigger óptico                                
DIV-501 -> EJE-602                      

### 8. $G_V$ — Grafo de Circulação de Veículos (Anel Viário de Sentido Único)
Cada sentido permitido é um arco explícito. Dois tipos de caminhão usam o anel: o **graneleiro** (traz arroz, para no calador e na moega) e o **caminhão de expedição** (chega vazio, faz a tara e segue direto para as docas pela via interna leste).

In [9]:
circulacao_veiculos = GrafoLogistico("G_V (circulação de veículos)")
adicionar_rotas(circulacao_veiculos, [
    ("Portaria", "Balança de entrada", 30, "entrada de veículos"),
    ("Balança de entrada", "Pátio do calador", 25, "fila de amostragem do graneleiro"),
    ("Pátio do calador", "Moega de descarga", 60, "lote liberado para descarga"),
    ("Pátio do calador", "Pátio de quarentena", 40, "lote aguardando laudo do LAB-CQ"),
    ("Pátio de quarentena", "Moega de descarga", 35, "laudo aprovado"),
    ("Pátio de quarentena", "Balança de saída", 120, "lote recusado e devolvido"),
    ("Moega de descarga", "Balança de saída", 110, "via oeste de retorno do graneleiro vazio"),
    ("Balança de entrada", "Via interna leste", 95, "caminhão de expedição após a tara"),
    ("Moega de descarga", "Via interna leste", 70, "acesso de serviço"),
    ("Via interna leste", "Docas de expedição", 55, "manobra de encoste"),
    ("Via interna leste", "Pátio de subprodutos", 45, "coleta de quirera"),
    ("Docas de expedição", "Balança de saída", 50, "pesagem do caminhão carregado"),
    ("Pátio de subprodutos", "Balança de saída", 65, "saída da caçamba"),
    ("Balança de saída", "Portaria", 35, "liberação de saída"),
])

viagens = {
    "Graneleiro (lote aprovado)":       ["Portaria", "Pátio do calador", "Moega de descarga", "Portaria"],
    "Graneleiro (lote em quarentena)":  ["Portaria", "Pátio do calador", "Pátio de quarentena", "Moega de descarga", "Portaria"],
    "Graneleiro (lote recusado)":       ["Portaria", "Pátio do calador", "Pátio de quarentena", "Portaria"],
    "Caminhão de expedição":            ["Portaria", "Docas de expedição", "Portaria"],
    "Caminhão de subprodutos":          ["Portaria", "Pátio de subprodutos", "Portaria"],
}
tabela_viagens = []
for nome, paradas in viagens.items():
    d, c = circulacao_veiculos.rota_com_paradas(paradas)
    tabela_viagens.append({"Viagem": nome, "Percurso": " -> ".join(c), "Distância (m)": f"{d:.0f}"})
print(formatar_tabela(tabela_viagens))

Viagem                          | Percurso                                                                                                                       | Distância (m)
--------------------------------+--------------------------------------------------------------------------------------------------------------------------------+--------------
Graneleiro (lote aprovado)      | Portaria -> Balança de entrada -> Pátio do calador -> Moega de descarga -> Balança de saída -> Portaria                        | 260          
Graneleiro (lote em quarentena) | Portaria -> Balança de entrada -> Pátio do calador -> Pátio de quarentena -> Moega de descarga -> Balança de saída -> Portaria | 275          
Graneleiro (lote recusado)      | Portaria -> Balança de entrada -> Pátio do calador -> Pátio de quarentena -> Balança de saída -> Portaria                      | 250          
Caminhão de expedição           | Portaria -> Balança de entrada -> Via interna leste -> Docas de expedição -> Bala

#### 8.1. Nenhum veículo fica preso: conectividade forte em relação à Portaria
Em um anel de sentido único, todo vértice precisa ser **alcançável a partir da Portaria** e **alcançar a Portaria** (BFS no grafo e no grafo reverso). Também verificamos que $G_V$, ao contrário de $G_M$, **contém ciclo** por construção.

In [10]:
def reverso(g: GrafoLogistico) -> GrafoLogistico:
    r = GrafoLogistico(g.nome + " reverso")
    for v in g.vertices:
        r._adjacencias.setdefault(v, [])
    for u, rota in g.arcos():
        r.adicionar_rota(rota.destino, u, rota.peso, rota.descricao)
    return r

ida = circulacao_veiculos.alcancaveis("Portaria")
volta = reverso(circulacao_veiculos).alcancaveis("Portaria")
print(formatar_tabela([{"Vértice": v, "Portaria alcança?": v in ida, "Alcança a Portaria?": v in volta}
                       for v in circulacao_veiculos.vertices]))
print(f"\nG_V fortemente conexo: {set(circulacao_veiculos.vertices) == ida == volta}")
print(f"Ciclo do anel viário: {' -> '.join(circulacao_veiculos.encontrar_ciclo())}")

Vértice              | Portaria alcança? | Alcança a Portaria?
---------------------+-------------------+--------------------
Portaria             | True              | True               
Balança de entrada   | True              | True               
Pátio do calador     | True              | True               
Moega de descarga    | True              | True               
Pátio de quarentena  | True              | True               
Balança de saída     | True              | True               
Via interna leste    | True              | True               
Docas de expedição   | True              | True               
Pátio de subprodutos | True              | True               

G_V fortemente conexo: True
Ciclo do anel viário: Portaria -> Balança de entrada -> Pátio do calador -> Moega de descarga -> Balança de saída -> Portaria


#### 8.2. Bloqueio dinâmico: moega interditada para limpeza
Com a via `Pátio do calador` $\to$ `Moega de descarga` interditada, o graneleiro só alcança a moega passando pelo pátio de quarentena.

In [11]:
gv_bloq = circulacao_veiculos.copia()
gv_bloq.remover_rota("Pátio do calador", "Moega de descarga")
d0, c0 = circulacao_veiculos.menor_caminho("Pátio do calador", "Moega de descarga")
d1, c1 = gv_bloq.menor_caminho("Pátio do calador", "Moega de descarga")
print(f"Nominal:   {' -> '.join(c0)} ({d0:.0f} m)")
print(f"Bloqueado: {' -> '.join(c1)} ({d1:.0f} m, +{d1 - d0:.0f} m)")

Nominal:   Pátio do calador -> Moega de descarga (60 m)
Bloqueado: Pátio do calador -> Pátio de quarentena -> Moega de descarga (75 m, +15 m)


### 9. $G_E$ — Alocação de Paletes no Galpão B (*Slotting*)
O custo de manuseio de um palete é a soma do **deslocamento de guarda** (paletizadora $\to$ posição) com o **deslocamento de separação** (posição $\to$ docas). Cada palete é alocado na posição livre de menor custo total; a posição ocupada sai do grafo (arco bloqueado), e o próximo palete é realocado.

In [12]:
estoque = GrafoLogistico("G_E (alocação de estoque)")
adicionar_rotas(estoque, [
    ("Paletizadora PAL-801", "Corredor 1", 8, "saída da paletizadora"),
    ("Paletizadora PAL-801", "Corredor 2", 15, "saída da paletizadora"),
    ("Corredor 1", "Posição A1", 6, "guarda"), ("Corredor 1", "Posição A2", 14, "guarda"), ("Corredor 1", "Posição A3", 22, "guarda"),
    ("Corredor 2", "Posição B1", 6, "guarda"), ("Corredor 2", "Posição B2", 14, "guarda"), ("Corredor 2", "Posição B3", 22, "guarda"),
    ("Posição A1", "Docas de expedição", 34, "separação"), ("Posição A2", "Docas de expedição", 24, "separação"),
    ("Posição A3", "Docas de expedição", 15, "separação"), ("Posição B1", "Docas de expedição", 27, "separação"),
    ("Posição B2", "Docas de expedição", 18, "separação"), ("Posição B3", "Docas de expedição", 9, "separação"),
])

def alocar_paletes(g: GrafoLogistico, quantidade: int) -> List[Dict[str, str]]:
    livre = g.copia()
    alocacoes = []
    for palete in range(1, quantidade + 1):
        candidatos = []
        for pos in [v for v in livre.vertices if v.startswith("Posição")]:
            try:
                guarda, _ = livre.menor_caminho("Paletizadora PAL-801", pos)
                separacao, _ = livre.menor_caminho(pos, "Docas de expedição")
            except ValueError:
                continue
            candidatos.append((guarda + separacao, guarda, separacao, pos))
        total, guarda, separacao, pos = min(candidatos)
        alocacoes.append({"Palete": palete, "Posição escolhida": pos, "Guarda (m)": guarda,
                          "Separação (m)": separacao, "Custo total (m)": total})
        corredor = "Corredor 1" if pos.endswith(("A1", "A2", "A3")) else "Corredor 2"
        livre.remover_rota(corredor, pos)   # posição ocupada: fica inacessível para guarda
    return alocacoes

alocacao = alocar_paletes(estoque, 4)
print(formatar_tabela(alocacao))

Palete | Posição escolhida | Guarda (m) | Separação (m) | Custo total (m)
-------+-------------------+------------+---------------+----------------
1      | Posição A3        | 30.0       | 15.0          | 45.0           
2      | Posição A2        | 22.0       | 24.0          | 46.0           
3      | Posição B3        | 37.0       | 9.0           | 46.0           
4      | Posição B2        | 29.0       | 18.0          | 47.0           


### 10. Rede Multicamada: Acoplamentos e Consultas na Camada Errada
Vértices com o mesmo espaço físico aparecem em camadas diferentes, mas **não são o mesmo vértice do grafo**. Uma consulta de circulação de veículo feita em $G_M$ (ou o contrário) deve ser rejeitada.

In [13]:
acoplamentos = [
    ("G_M", "Calador / amostragem", "G_V", "Pátio do calador"),
    ("G_M", "Moega MG-001", "G_V", "Moega de descarga"),
    ("G_M", "Caçamba CAC-803", "G_V", "Pátio de subprodutos"),
    ("G_M", "Docas de expedição", "G_V", "Docas de expedição"),
    ("G_M", "Paletizadora PAL-801", "G_E", "Paletizadora PAL-801"),
    ("G_M", "Estoque Tipo 1", "G_E", "Posição A1 .. B3"),
]
print(formatar_tabela([{"Camada α": a, "Vértice em α": va, "Camada β": b, "Vértice em β": vb}
                       for a, va, b, vb in acoplamentos]))

print("\nConsulta na camada errada: rota de caminhão até o SILO-701 em G_V")
try:
    circulacao_veiculos.menor_caminho("Portaria", "SILO-701")
except KeyError as erro:
    print(f"  Rejeitada corretamente -> {erro}")

print("Consulta na camada errada: grão da Moega até a Via interna leste em G_M")
try:
    fluxo_materiais.menor_caminho("Moega MG-001", "Via interna leste")
except KeyError as erro:
    print(f"  Rejeitada corretamente -> {erro}")

Camada α | Vértice em α         | Camada β | Vértice em β        
---------+----------------------+----------+---------------------
G_M      | Calador / amostragem | G_V      | Pátio do calador    
G_M      | Moega MG-001         | G_V      | Moega de descarga   
G_M      | Caçamba CAC-803      | G_V      | Pátio de subprodutos
G_M      | Docas de expedição   | G_V      | Docas de expedição  
G_M      | Paletizadora PAL-801 | G_E      | Paletizadora PAL-801
G_M      | Estoque Tipo 1       | G_E      | Posição A1 .. B3    

Consulta na camada errada: rota de caminhão até o SILO-701 em G_V
  Rejeitada corretamente -> 'Origem e destino devem pertencer ao grafo G_V (circulação de veículos).'
Consulta na camada errada: grão da Moega até a Via interna leste em G_M
  Rejeitada corretamente -> 'Origem e destino devem pertencer ao grafo G_M (fluxo de materiais).'


### 11. Asserções e Validações Formais de Engenharia

In [14]:
# Validação 1: a rota de referência passa pela linha de classificação e pela inspeção óptica
assert "FN-101" in caminho and "VIS-401" in caminho, "Falha: todo produto deve passar pela visão computacional!"

# Validação 2: Dijkstra soma o caminho acumulado (SP-002: 42 + 20 = 62 < SP-001: 35 + 30 = 65)
assert "Silo pulmão SP-002" in caminho and abs(distancia - 261.3) < 1e-6, "Falha na rota de referência!"

# Validação 3: segregação sanitária do rejeito
assert not set(proibidos) & alcance_rejeito, "Falha: rejeito alcança área de produto acabado!"

# Validação 4: G_M sem reciclo é DAG; com reciclo, o ciclo detectado é o de e14
assert not restantes and restantes_r, "Falha na detecção de aciclicidade!"
assert ciclo[0] == ciclo[-1] and "SILO-702" in ciclo and "FN-101" in ciclo, "Falha: ciclo deve conter e14!"

# Validação 5: sequências fisicamente impossíveis são rejeitadas
assert [fluxo_materiais.sequencia_fisicamente_valida(s) for s in sequencias.values()] == [True, False, True, False]

# Validação 6: redundância dos silos pulmão (TC-001/TC-002 não são críticos) e linha única é crítica
nomes_criticos = {(u, r.destino) for u, r in criticos}
assert ("Moega MG-001", "Silo pulmão SP-002") not in nomes_criticos and ("BAL-301", "VIS-401") in nomes_criticos

# Validação 7: G_V fortemente conexo e com ciclo; caminhão de expedição não passa pela moega
assert set(circulacao_veiculos.vertices) == ida == volta and circulacao_veiculos.encontrar_ciclo()
assert "Moega de descarga" not in circulacao_veiculos.rota_com_paradas(viagens["Caminhão de expedição"])[1]

# Validação 8: alocação de estoque não repete posição
assert len({a["Posição escolhida"] for a in alocacao}) == len(alocacao)

print("Todas as 8 asserções matemáticas e operacionais foram VALIDADAS com sucesso!")

Todas as 8 asserções matemáticas e operacionais foram VALIDADAS com sucesso!
